# GPU Memory Test: Max Batch Size on T4

Find the maximum `per_device_train_batch_size` for ModernBERT-base + LoRA (same config as NB01) across different sequence lengths on T4 (16GB).

Tests forward + backward pass with gradient checkpointing to match real training memory usage.

In [ ]:
%%capture
import os, re, torch
v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
!pip install sentencepiece protobuf "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2

In [ ]:
%env UNSLOTH_DISABLE_FAST_GENERATION=1

import torch
import gc
import time
from unsloth import FastModel, is_bfloat16_supported
from transformers import AutoModelForSequenceClassification

NUM_CLASSES = 3
ID_TO_LABEL = {0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"}
LABEL_TO_ID = {"NEGATIVE": 0, "NEUTRAL": 1, "POSITIVE": 2}
MAX_LENGTH = 8192
MODEL_NAME = "unsloth/ModernBERT-base"
SEED = 3407

cap = torch.cuda.get_device_capability()
assert cap[0] >= 7, f"GPU sm_{cap[0]}{cap[1]} not supported — need sm_70+ (T4/V100/A100)"
name = torch.cuda.get_device_name()
props = torch.cuda.get_device_properties(0)
mem_gb = getattr(props, 'total_memory', getattr(props, 'total_mem', 0)) / 1e9
print(f"GPU: {name} (sm_{cap[0]}{cap[1]}, {mem_gb:.1f} GB)")

In [ ]:
# Load model + LoRA — identical config to NB01
model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    auto_model=AutoModelForSequenceClassification,
    max_seq_length=MAX_LENGTH,
    dtype=None,
    num_labels=NUM_CLASSES,
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
    load_in_4bit=False,
)

model = FastModel.get_peft_model(
    model,
    r=16,
    target_modules=["Wqkv", "out_proj", "Wi", "Wo"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
    task_type="SEQ_CLS",
)

print(f"Model loaded on {next(model.parameters()).device}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def gpu_mem_mb(device=0):
    """Current allocated GPU memory in MB."""
    return torch.cuda.memory_allocated(device) / 1e6

def gpu_peak_mb(device=0):
    """Peak allocated GPU memory in MB."""
    return torch.cuda.max_memory_allocated(device) / 1e6

def clear_gpu():
    """Aggressively free GPU memory."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

def test_batch(model, seq_len, batch_size, device=0):
    """
    Run a forward + backward pass with dummy data.
    Returns (peak_mb, time_sec) or raises OOM.
    """
    clear_gpu()
    
    input_ids = torch.randint(1, 30000, (batch_size, seq_len), device=f"cuda:{device}")
    attention_mask = torch.ones_like(input_ids)
    labels = torch.randint(0, NUM_CLASSES, (batch_size,), device=f"cuda:{device}")
    
    torch.cuda.reset_peak_memory_stats(device)
    
    t0 = time.time()
    with torch.amp.autocast("cuda", dtype=torch.float16):
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
    loss.backward()
    torch.cuda.synchronize()
    elapsed = time.time() - t0
    
    peak = gpu_peak_mb(device)
    
    # Clean up
    model.zero_grad(set_to_none=True)
    del input_ids, attention_mask, labels, outputs, loss
    clear_gpu()
    
    return peak, elapsed

print("Memory test functions ready.")

In [ ]:
# --- Main profiling loop ---
# Test sequence lengths that match the actual data distribution:
#   Short-context: median ~44 tokens, max ~512
#   Long-context: 512-8192 tokens, median ~1500
SEQ_LENGTHS = [64, 128, 256, 512, 1024, 2048, 4096, 8192]
BATCH_SIZES = [1, 2, 4, 8, 16, 32, 64]

results = []
device = 0

model.train()
baseline_mb = gpu_mem_mb(device)
print(f"Baseline GPU memory (model loaded): {baseline_mb:.0f} MB")
print(f"GPU total: {mem_gb:.1f} GB")
print(f"\n{'SeqLen':>8} {'Batch':>6} {'Peak MB':>10} {'Time (s)':>10} {'Status':>10}")
print("-" * 52)

for seq_len in SEQ_LENGTHS:
    max_ok_batch = 0
    for bs in BATCH_SIZES:
        try:
            peak, elapsed = test_batch(model, seq_len, bs, device)
            status = "OK"
            max_ok_batch = bs
            results.append({
                "seq_len": seq_len, "batch_size": bs,
                "peak_mb": peak, "time_s": elapsed, "oom": False
            })
            print(f"{seq_len:>8} {bs:>6} {peak:>10.0f} {elapsed:>10.2f} {status:>10}")
        except torch.cuda.OutOfMemoryError:
            clear_gpu()
            results.append({
                "seq_len": seq_len, "batch_size": bs,
                "peak_mb": -1, "time_s": -1, "oom": True
            })
            print(f"{seq_len:>8} {bs:>6} {'—':>10} {'—':>10} {'OOM':>10}")
            # No point testing larger batch sizes at this seq_len
            for bs2 in BATCH_SIZES[BATCH_SIZES.index(bs)+1:]:
                results.append({
                    "seq_len": seq_len, "batch_size": bs2,
                    "peak_mb": -1, "time_s": -1, "oom": True
                })
            break

print("\nDone.")

In [ ]:
# --- Summary: max batch size per sequence length ---
import pandas as pd

df = pd.DataFrame(results)
summary = df[~df["oom"]].groupby("seq_len").agg(
    max_batch=("batch_size", "max"),
    peak_mb_at_max=("peak_mb", "last"),
    time_at_max=("time_s", "last"),
).reset_index()

print("=" * 60)
print("RESULTS: Max batch size per sequence length (T4 16GB)")
print("=" * 60)
print(f"\n{'SeqLen':>8} {'MaxBatch':>10} {'PeakMem':>12} {'Time/step':>12}")
print("-" * 46)
for _, row in summary.iterrows():
    print(f"{int(row['seq_len']):>8} {int(row['max_batch']):>10} {row['peak_mb_at_max']:>10.0f} MB {row['time_at_max']:>10.2f}s")

print("\n--- Training time estimates (79K rows, 3 epochs) ---")
for _, row in summary.iterrows():
    sl = int(row["seq_len"])
    bs = int(row["max_batch"])
    t = row["time_at_max"]
    if sl in [64, 2048, 8192]:
        steps_per_epoch = 79_000 / bs
        total_time_h = (steps_per_epoch * 3 * t) / 3600
        print(f"  seq_len={sl}, batch={bs}: ~{total_time_h:.1f} hours for 79K×3ep")

print("\nWith dynamic padding + group_by_length, short sequences use larger batches.")

In [ ]:
# --- Also test with load_in_4bit=True for comparison ---
print("Testing 4-bit quantization at key sequence lengths...")
print("(Reloading model with load_in_4bit=True)\n")

del model
clear_gpu()

model_4bit, _ = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    auto_model=AutoModelForSequenceClassification,
    max_seq_length=MAX_LENGTH,
    dtype=None,
    num_labels=NUM_CLASSES,
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
    load_in_4bit=True,
)

model_4bit = FastModel.get_peft_model(
    model_4bit,
    r=16,
    target_modules=["Wqkv", "out_proj", "Wi", "Wo"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
    task_type="SEQ_CLS",
)

model_4bit.train()
baseline_4bit = gpu_mem_mb(0)
print(f"4-bit baseline memory: {baseline_4bit:.0f} MB (vs fp16: {baseline_mb:.0f} MB)")

KEY_LENGTHS = [512, 2048, 4096, 8192]
results_4bit = []

print(f"\n{'SeqLen':>8} {'Batch':>6} {'Peak MB':>10} {'Time (s)':>10} {'Status':>10}")
print("-" * 52)

for seq_len in KEY_LENGTHS:
    for bs in BATCH_SIZES:
        try:
            peak, elapsed = test_batch(model_4bit, seq_len, bs, 0)
            results_4bit.append({
                "seq_len": seq_len, "batch_size": bs,
                "peak_mb": peak, "time_s": elapsed, "oom": False
            })
            print(f"{seq_len:>8} {bs:>6} {peak:>10.0f} {elapsed:>10.2f} {'OK':>10}")
        except torch.cuda.OutOfMemoryError:
            clear_gpu()
            results_4bit.append({
                "seq_len": seq_len, "batch_size": bs,
                "peak_mb": -1, "time_s": -1, "oom": True
            })
            print(f"{seq_len:>8} {bs:>6} {'—':>10} {'—':>10} {'OOM':>10}")
            for bs2 in BATCH_SIZES[BATCH_SIZES.index(bs)+1:]:
                results_4bit.append({
                    "seq_len": seq_len, "batch_size": bs2,
                    "peak_mb": -1, "time_s": -1, "oom": True
                })
            break

del model_4bit
clear_gpu()

In [ ]:
# --- Final comparison ---
df_4bit = pd.DataFrame(results_4bit)
summary_4bit = df_4bit[~df_4bit["oom"]].groupby("seq_len").agg(
    max_batch=("batch_size", "max"),
    peak_mb_at_max=("peak_mb", "last"),
).reset_index()

print("=" * 65)
print("COMPARISON: fp16 vs 4-bit max batch size (T4 16GB)")
print("=" * 65)
print(f"\n{'SeqLen':>8} {'fp16 Batch':>12} {'fp16 Peak':>12} {'4bit Batch':>12} {'4bit Peak':>12}")
print("-" * 60)

for sl in KEY_LENGTHS:
    fp16_row = summary[summary["seq_len"] == sl]
    q4_row = summary_4bit[summary_4bit["seq_len"] == sl]
    fp16_bs = int(fp16_row["max_batch"].iloc[0]) if len(fp16_row) else 0
    fp16_pk = f"{fp16_row['peak_mb_at_max'].iloc[0]:.0f}" if len(fp16_row) else "OOM"
    q4_bs = int(q4_row["max_batch"].iloc[0]) if len(q4_row) else 0
    q4_pk = f"{q4_row['peak_mb_at_max'].iloc[0]:.0f}" if len(q4_row) else "OOM"
    print(f"{sl:>8} {fp16_bs:>12} {fp16_pk:>10} MB {q4_bs:>12} {q4_pk:>10} MB")

print("\nUse these results to set per_device_train_batch_size in NB01.")
print("For mixed-length training with dynamic padding, use the SMALLEST")
print("max batch (from longest seq_len) as a safe default, or use")
print("group_by_length=True to batch similar lengths together.")